# 03 - Modelling

Compare simple/statistical forecasts with LightGBM and CatBoost, then fit market-regime, anomaly, and change-point models. Preprocessing is fitted on training data only. The optional deep-learning wrappers are outside this core run.

In [1]:
## Setup

import os
os.environ.pop("SPARK_HOME", None)  # Use the JARs bundled with the installed PySpark version.

import sys, warnings, yaml, joblib
from pathlib import Path
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

_cwd = Path.cwd().resolve()
_candidates = list(dict.fromkeys([
    _cwd,
    *_cwd.parents,
    *(path.parent.parent for path in _cwd.glob("*/configs/pipeline.yaml")),
]))
_required_project_files = (
    "configs/pipeline.yaml",
    "src/features/pipeline.py",
    "notebooks/01_data_pipeline.ipynb",
    "Home.py",
)
_project_roots = [
    path for path in _candidates
    if all((path / relative).is_file() for relative in _required_project_files)
]
if not _project_roots:
    raise FileNotFoundError(
        "Could not identify this project. Open its repository root or set the notebook working directory to it."
    )
if len(_project_roots) > 1:
    raise RuntimeError(
        f"Multiple project roots found: {_project_roots}. Open the intended repository root."
    )
PROJECT_ROOT = _project_roots[0]
os.chdir(PROJECT_ROOT)
CONFIG_PATH = PROJECT_ROOT / "configs" / "pipeline.yaml"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.baselines       import naive_persistence, fit_arima, fit_garch
from src.models.lgbm_model      import train_lgbm
from src.models.catboost_model  import train_catboost
from src.models.regime_models   import fit_gmm, fit_hmm
from src.models.anomaly_models  import fit_isolation_forest, compute_matrix_profile, detect_changepoints_cusum
from src.visualisation.model_plots  import plot_forecast_vs_actual, plot_feature_importance
from src.visualisation.regime_plots import plot_regime_states, plot_volatility_regimes

with open(PROJECT_ROOT / 'configs' / 'pipeline.yaml') as f:
    CFG = yaml.safe_load(f)

PROCESSED = PROJECT_ROOT / CFG['artifacts']['processed_dir']
MODELS    = PROJECT_ROOT / CFG['artifacts']['models_dir']
SCALERS   = PROJECT_ROOT / CFG['artifacts']['scalers_dir']
FIGURES   = PROJECT_ROOT / CFG['reports']['figures_dir']
METRICS   = PROJECT_ROOT / CFG['reports']['metrics_dir']
for d in [MODELS, SCALERS, FIGURES, METRICS]:
    d.mkdir(parents=True, exist_ok=True)

with open(PROCESSED / 'split_meta.yaml') as f:
    SPLIT = yaml.safe_load(f)

FEATURE_COLS = SPLIT['feature_cols']
TARGET_COL   = SPLIT['target_col']
TS_COL       = SPLIT['ts_col']

print(f'Features : {len(FEATURE_COLS)}  |  Target : {TARGET_COL}')

Features : 62  |  Target : target_log_return_1


In [2]:
# ── Load partitions ───────────────────────────────────────────────────
train = pd.read_parquet(PROCESSED / 'train.parquet')
val = pd.read_parquet(PROCESSED / 'val.parquet')
test = pd.read_parquet(PROCESSED / 'test.parquet')
for partition in (train, val, test):
    partition[TS_COL] = pd.to_datetime(partition[TS_COL], utc=True)
    partition.sort_values(TS_COL, inplace=True)
    partition.reset_index(drop=True, inplace=True)

X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_val,   y_val   = val[FEATURE_COLS],   val[TARGET_COL]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET_COL]

print(f'Train {X_train.shape}  |  Val {X_val.shape}  |  Test {X_test.shape}')

Train (558601, 62)  |  Val (119999, 62)  |  Test (120000, 62)


In [3]:
# ── Shared metric function ────────────────────────────────────────────
RESULTS = {}  # keyed by model/partition so rerunning a cell replaces its result

def compute_metrics(y_true, y_pred, model='', partition=''):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    if len(y_true) != len(y_pred):
        raise ValueError(f"{model} {partition}: actual/prediction lengths differ.")
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if not mask.any():
        raise ValueError(f"{model} {partition}: no finite actual/prediction pairs.")
    y_true, y_pred = y_true[mask], y_pred[mask]
    mae  = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    smape = float(np.mean(np.where(denom==0, 0, np.abs(y_true-y_pred)/denom)) * 100)
    result = {'model': model, 'partition': partition,
              'mae': mae, 'rmse': rmse, 'smape': smape}
    RESULTS[(model, partition)] = result
    print(f'  [{model}] [{partition}]  MAE={mae:.6f}  RMSE={rmse:.6f}  sMAPE={smape:.2f}%')
    return result

---
## A · Baselines
### A1 · Naïve / Persistence

In [4]:
# Predict: last observed return value
target_history = pd.concat([train[TARGET_COL], val[TARGET_COL], test[TARGET_COL]], ignore_index=True)
naive_predictions = naive_persistence(target_history, horizon=1)
val_start, test_start = len(train), len(train) + len(val)
naive_val = naive_predictions.iloc[val_start:test_start]
naive_test = naive_predictions.iloc[test_start:]
compute_metrics(y_val, naive_val, model='Naive', partition='Val')
compute_metrics(y_test, naive_test, model='Naive', partition='Test')

  [Naive] [Val]  MAE=0.000795  RMSE=0.001728  sMAPE=147.57%
  [Naive] [Test]  MAE=0.001041  RMSE=0.002849  sMAPE=148.26%


C:\Users\hp\AppData\Local\Temp\ipykernel_9552\2003014966.py:15: RuntimeWarning: invalid value encountered in divide
  smape = float(np.mean(np.where(denom==0, 0, np.abs(y_true-y_pred)/denom)) * 100)
C:\Users\hp\AppData\Local\Temp\ipykernel_9552\2003014966.py:15: RuntimeWarning: invalid value encountered in divide
  smape = float(np.mean(np.where(denom==0, 0, np.abs(y_true-y_pred)/denom)) * 100)


{'model': 'Naive',
 'partition': 'Test',
 'mae': 0.001041400411554967,
 'rmse': 0.0028487400686007854,
 'smape': 148.26420611651997}

### A2 · ARIMA(1,0,1)

In [5]:
arima_series = train['log_return'].dropna()
print(f'Fitting ARIMA(1,0,1) on {len(arima_series):,} training obs...')
arima_res = fit_arima(arima_series, order=(1,0,1), forecast_steps=len(y_val)+len(y_test))
print(f'AIC={arima_res["aic"]:.2f}  BIC={arima_res["bic"]:.2f}')

fc = arima_res['forecast'].values
compute_metrics(y_val,  fc[:len(y_val)],                      model='ARIMA(1,0,1)', partition='Val')
compute_metrics(y_test, fc[len(y_val):len(y_val)+len(y_test)], model='ARIMA(1,0,1)', partition='Test')

Fitting ARIMA(1,0,1) on 558,601 training obs...


AIC=-5362405.47  BIC=-5362360.53
  [ARIMA(1,0,1)] [Val]  MAE=0.000538  RMSE=0.001218  sMAPE=196.74%
  [ARIMA(1,0,1)] [Test]  MAE=0.000704  RMSE=0.002021  sMAPE=197.27%


{'model': 'ARIMA(1,0,1)',
 'partition': 'Test',
 'mae': 0.0007037274320757253,
 'rmse': 0.00202086499441617,
 'smape': 197.2686983472246}

### A3 · GARCH(1,1)

In [6]:
garch_series = train['log_return'].dropna()
print(f'Fitting GARCH(1,1) on {len(garch_series):,} training obs...')
garch_res = fit_garch(garch_series,
                      p=CFG['models']['garch']['p'],
                      q=CFG['models']['garch']['q'],
                      dist=CFG['models']['garch']['dist'],
                      forecast_horizon=1)
print(f'AIC={garch_res["aic"]:.2f}  BIC={garch_res["bic"]:.2f}')
print(f'1-step volatility forecast: {float(garch_res["variance_forecast"][0]):.6f}')

# Save conditional vol for Streamlit
cond_vol = garch_res['conditional_vol']
cv_index = train['log_return'].dropna().index
cv_df = pd.DataFrame({TS_COL: train.loc[cv_index, TS_COL].values,
                      'cond_vol': cond_vol.values[:len(cv_index)]})
cv_df.to_csv(MODELS / 'garch_cond_vol.csv', index=False)
print('GARCH conditional vol saved.')


Fitting GARCH(1,1) on 558,601 training obs...


c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\arch\univariate\base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.0399. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)


AIC=-581392.34  BIC=-581347.41
1-step volatility forecast: 0.006556
GARCH conditional vol saved.


---
## B · ML Models
### B1 · LightGBM

In [7]:
print('Training LightGBM...')
lgbm_res   = train_lgbm(X_train, y_train, X_val, y_val,
                         task='regression', artifact_name='lgbm_return')
lgbm_model  = lgbm_res['model']
lgbm_scaler = lgbm_res['scaler']

lgbm_val_pred  = lgbm_model.predict(lgbm_scaler.transform(X_val))
lgbm_test_pred = lgbm_model.predict(lgbm_scaler.transform(X_test))
compute_metrics(y_val,  lgbm_val_pred,  model='LightGBM', partition='Val')
compute_metrics(y_test, lgbm_test_pred, model='LightGBM', partition='Test')

fig = plot_forecast_vs_actual(y_test, pd.Series(lgbm_test_pred, index=y_test.index),
                              index=test.loc[y_test.index, TS_COL],
                              title='LightGBM — Test Forecast vs Actual')
fig.write_html(str(FIGURES / 'lgbm_forecast.html'), include_plotlyjs='cdn')

fig_imp = plot_feature_importance(FEATURE_COLS, lgbm_model.feature_importances_,
                                  top_n=25, title='LightGBM — Feature Importance')
fig_imp.write_html(str(FIGURES / 'lgbm_feature_importance.html'), include_plotlyjs='cdn')


Training LightGBM...
  LightGBM artifact saved → artifacts\models\lgbm_return.pkl


c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  [LightGBM] [Val]  MAE=0.000538  RMSE=0.001217  sMAPE=193.07%
  [LightGBM] [Test]  MAE=0.000705  RMSE=0.002025  sMAPE=192.91%


### B2 · CatBoost

In [8]:
print('Training CatBoost...')
cb_res   = train_catboost(X_train, y_train, X_val, y_val,
                           task='regression', artifact_name='catboost_return')
cb_model  = cb_res['model']
cb_scaler = cb_res['scaler']

cb_val_pred  = cb_model.predict(cb_scaler.transform(X_val))
cb_test_pred = cb_model.predict(cb_scaler.transform(X_test))
compute_metrics(y_val,  cb_val_pred,  model='CatBoost', partition='Val')
compute_metrics(y_test, cb_test_pred, model='CatBoost', partition='Test')

fig = plot_forecast_vs_actual(y_test, pd.Series(cb_test_pred, index=y_test.index),
                              index=test.loc[y_test.index, TS_COL],
                              title='CatBoost — Test Forecast vs Actual')
fig.write_html(str(FIGURES / 'catboost_forecast.html'), include_plotlyjs='cdn')


Training CatBoost...
  CatBoost artifact saved → artifacts\models\catboost_return.pkl
  [CatBoost] [Val]  MAE=0.000538  RMSE=0.001217  sMAPE=187.88%
  [CatBoost] [Test]  MAE=0.000704  RMSE=0.002021  sMAPE=188.49%


---
## D · Regime Discovery
### D1 · GMM + HMM

In [9]:
# ── Regime feature set (vol + return — good discriminators) ──────────
regime_fcols = [c for c in FEATURE_COLS
                if any(x in c for x in ('rolling_vol','log_return','momentum','rolling_return'))]
print(f'Regime features ({len(regime_fcols)}): {regime_fcols}')

X_reg_train = train[regime_fcols].dropna().values
X_reg_full  = pd.read_parquet(PROCESSED / 'btc_features.parquet')[regime_fcols].dropna().values

regime_scaler = StandardScaler()
X_reg_train_s = regime_scaler.fit_transform(X_reg_train)
X_reg_full_s  = regime_scaler.transform(X_reg_full)
joblib.dump(regime_scaler, SCALERS / 'regime_scaler.pkl')
print('Regime scaler fitted on training data only.')

Regime features (21): ['log_return', 'rolling_return_7', 'rolling_return_14', 'rolling_return_30', 'rolling_return_60', 'rolling_vol_7', 'rolling_vol_ann_7', 'rolling_vol_14', 'rolling_vol_ann_14', 'rolling_vol_30', 'rolling_vol_ann_30', 'rolling_vol_60', 'rolling_vol_ann_60', 'rolling_volume_7', 'rolling_volume_14', 'rolling_volume_30', 'rolling_volume_60', 'momentum_7', 'momentum_14', 'momentum_30', 'momentum_60']
Regime scaler fitted on training data only.


In [10]:
# ── GMM ───────────────────────────────────────────────────────────────
print('Fitting GMM...')
gmm_res    = fit_gmm(X_reg_train_s, artifact_name='gmm')
gmm_states = gmm_res['model'].predict(X_reg_full_s)

df_full = pd.read_parquet(PROCESSED / 'btc_features.parquet').sort_values(TS_COL).reset_index(drop=True)
df_full[TS_COL] = pd.to_datetime(df_full[TS_COL])
regime_index    = df_full[regime_fcols].dropna().index
df_full.loc[regime_index, 'gmm_regime'] = gmm_states.astype(int)

print(pd.Series(gmm_states).value_counts().sort_index().rename('bars_in_regime'))

fig = plot_regime_states(df_full.loc[regime_index], ts_col=TS_COL,
                         state_col='gmm_regime', title='GMM Market Regimes')
fig.write_html(str(FIGURES / 'gmm_regimes.html'), include_plotlyjs='cdn')


Fitting GMM...
  GMM artifact saved → artifacts\models\gmm.pkl
0    309407
1    390191
2    100399
Name: bars_in_regime, dtype: int64


In [11]:
# ── HMM ───────────────────────────────────────────────────────────────
print('Fitting HMM...')
hmm_res    = fit_hmm(X_reg_train_s, artifact_name='hmm')
hmm_states = hmm_res['model'].predict(X_reg_full_s)
df_full.loc[regime_index, 'hmm_state'] = hmm_states.astype(int)

print(pd.Series(hmm_states).value_counts().sort_index().rename('bars_in_state'))
print('\nHMM Transition Matrix:')
n = hmm_res['model'].n_components
print(pd.DataFrame(hmm_res['model'].transmat_,
                   columns=[f'→S{i}' for i in range(n)],
                   index=[f'S{i}→' for i in range(n)]).round(3))

fig = plot_regime_states(df_full.loc[regime_index], ts_col=TS_COL,
                         state_col='hmm_state', title='HMM Market States')
fig.write_html(str(FIGURES / 'hmm_states.html'), include_plotlyjs='cdn')


Fitting HMM...


Model is not converging.  Current: 10231758.628961025 is not greater than 25097323.99906856. Delta is -14865565.370107533


  HMM artifact saved → artifacts\models\hmm.pkl
1    278485
2    521512
Name: bars_in_state, dtype: int64

HMM Transition Matrix:
     →S0    →S1    →S2
S0→  0.0  0.000  1.000
S1→  0.0  0.975  0.025
S2→  0.0  0.005  0.995


---
## E · Anomaly & Change-Point Detection
### E1 · Isolation Forest

In [12]:
print('Fitting Isolation Forest...')
X_anom_train = train[FEATURE_COLS].dropna().values
X_anom_full  = df_full[FEATURE_COLS].dropna().values

anom_scaler   = StandardScaler()
X_anom_tr_s   = anom_scaler.fit_transform(X_anom_train)
X_anom_full_s = anom_scaler.transform(X_anom_full)
joblib.dump(anom_scaler, SCALERS / 'anomaly_scaler.pkl')

iso_res = fit_isolation_forest(X_anom_tr_s, X_anom_full_s, artifact_name='isolation_forest')

anom_index = df_full[FEATURE_COLS].dropna().index
scores_df  = pd.DataFrame({
    TS_COL:       df_full.loc[anom_index, TS_COL].values,
    'score':      iso_res['full_scores'],
    'is_anomaly': iso_res['anomaly_flags'].astype(int),
})
scores_df.to_csv(MODELS / 'anomaly_scores.csv', index=False)
print(f'Anomalies flagged: {iso_res["anomaly_flags"].sum():,} '
      f'({iso_res["anomaly_flags"].mean()*100:.2f}%)')

fig = go.Figure([
    go.Scatter(x=pd.to_datetime(scores_df[TS_COL]), y=scores_df['score'],
               name='IF Score', line=dict(color='#4FC3F7', width=0.8)),
    go.Scatter(x=pd.to_datetime(scores_df[scores_df['is_anomaly']==1][TS_COL]),
               y=scores_df[scores_df['is_anomaly']==1]['score'],
               mode='markers', name='Anomaly',
               marker=dict(color='#EF5350', size=5, symbol='x')),
])
fig.update_layout(title='Isolation Forest — Anomaly Scores', template='plotly_dark')
fig.write_html(str(FIGURES / 'anomaly_scores.html'), include_plotlyjs='cdn')


Fitting Isolation Forest...
  Isolation Forest artifact saved → artifacts\models\isolation_forest.pkl
Anomalies flagged: 14,391 (1.80%)


### E2 · Matrix Profile

In [13]:
print('Computing Matrix Profile (may take a few minutes)...')

# Use training log_return series only (correct partition for unsupervised discovery).
# Cap at 50 000 bars — stumpy.stump() is O(n²); beyond this it becomes very slow.
MP_MAX_ROWS = 50_000
lr_series = train['log_return'].dropna().iloc[-MP_MAX_ROWS:].values
mp_index  = train['log_return'].dropna().iloc[-MP_MAX_ROWS:].index

print(f'  Series length : {len(lr_series):,} bars (window=30)')
mp_res   = compute_matrix_profile(lr_series, window=30)
disc_idx = mp_res['discord_idx']

disc_ts = train.loc[mp_index[disc_idx], TS_COL] if disc_idx < len(mp_index) else 'N/A'
print(f'Discord (most unusual 30-bar window) at index {disc_idx}  →  {disc_ts}')
print(f'Discord MP value: {mp_res["discord_value"]:.4f}')

mp_ts = pd.to_datetime(train.loc[mp_index[:len(mp_res["mp"])], TS_COL].values)
fig = go.Figure([go.Scatter(x=mp_ts, y=mp_res['mp'], name='MP', line=dict(color='#F7931A'))])
if disc_idx < len(mp_ts):
    discord_ts = mp_ts[disc_idx]
    fig.add_vline(x=discord_ts, line_color='#EF5350', line_dash='dot')
    fig.add_annotation(x=discord_ts, y=1, yref='paper', text='Discord',
                       showarrow=False, yshift=8)
fig.update_layout(title='Matrix Profile — Log Returns (window=30)', template='plotly_dark')
fig.write_html(str(FIGURES / 'matrix_profile.html'), include_plotlyjs='cdn')


Computing Matrix Profile (may take a few minutes)...
  Series length : 50,000 bars (window=30)
Discord (most unusual 30-bar window) at index 13061  →  2019-06-03 10:38:00+00:00
Discord MP value: 4.9036


### E3 · CUSUM Change-Point Detection

In [14]:
# ⚠️  Calibrated on training partition only
cusum_series = train['log_return'].dropna().values
print(f'Detecting change-points on {len(cusum_series):,} training observations (RBF cap: 3,000)...')

bkps = detect_changepoints_cusum(
    cusum_series, n_bkps=5, model='rbf', max_samples=3_000,
)

train_ts_index = train['log_return'].dropna().index
bp_records = []
for bp in bkps[:-1]:
    if bp < len(train_ts_index):
        ts_val = train.loc[train_ts_index[bp], TS_COL]
        bp_records.append({'index': bp, 'timestamp': str(ts_val)})
        print(f'  Breakpoint {bp} → {ts_val}')

pd.DataFrame(bp_records).to_csv(MODELS / 'changepoints.csv', index=False)

vol_col = next((c for c in FEATURE_COLS if 'rolling_vol_' in c), 'log_return')
fig = plot_volatility_regimes(
    train,
    vol_col=vol_col, ts_col=TS_COL,
    changepoints=[r['index'] for r in bp_records] + [len(train)],
    title='CUSUM Change-Points on Rolling Volatility',
)
fig.write_html(str(FIGURES / 'cusum_changepoints.html'), include_plotlyjs='cdn')


Detecting change-points on 558,601 training observations (RBF cap: 3,000)...
  Breakpoint 66385 → 2017-11-10 10:37:00+00:00
  Breakpoint 216920 → 2018-05-18 12:23:00+00:00
  Breakpoint 502095 → 2019-05-10 02:16:00+00:00


In [15]:
# ── Save interim metrics for notebook 04 ─────────────────────────────
metrics_path = PROCESSED / 'metrics_draft.yaml'
with open(metrics_path, 'w') as f:
    yaml.safe_dump(list(RESULTS.values()), f, sort_keys=False)
print(f'Interim metrics saved → {metrics_path}')
print('Next → 04_evaluation_and_export.ipynb')

Interim metrics saved → C:\Users\hp\Documents\GitHub\Crypto_TimeSeries_Analysis\processed\metrics_draft.yaml
Next → 04_evaluation_and_export.ipynb
